In [11]:
# general imports
import pandas as pd
import os
import json

In [ ]:
# Optional environment setup.
# Set these before initializing vLLM, PyTorch, sentence-transformers, or transformers models.

# Restrict this notebook process to specific GPUs.
# Examples: "0" for one GPU, "0,1" for two GPUs.
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Optional: set for higher Hugging Face Hub rate limits and faster first-time downloads.
# os.environ["HF_TOKEN"] = "hf_..."

# MMAI Examples

## 1. Trial summarization walkthrough


### Read in data

In [12]:
trials_path = "data/scheduled__2025-09-04T230000+0000.trials_for_summarize.csv"
trials = pd.read_csv(trials_path)

### Local mode

`summarize_trials(...)` uses the local in-memory vLLM backend by default. Run this cell if you want local mode.


In [ ]:
from matchminer_ai.trials import summarize_trials

trial_results, metadata, qc_report = summarize_trials(
    trials, return_metadata=True, return_qc=True
)

# Optional: clear the cached local vLLM engine if GPU memory is tight.
# from matchminer_ai.llm.backends import clear_local_llm_cache
# clear_local_llm_cache()

### Remote mode

Run this cell instead of the local-mode cell if you want to send trial summarization requests to an existing OpenAI-compatible endpoint. If you do not already have a vLLM server running, `start_vllm_server(...)` can start one from the config values.

In [13]:
from matchminer_ai import load_preset
from matchminer_ai.trials import summarize_trials

os.environ["OPENAI_API_KEY"] = "not-needed"

# Optional: start a local vLLM server if you do not already have one.
# Optional: choose GPUs for the server process before starting it.
# from matchminer_ai.llm.vllm_server import start_vllm_server
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
# process = start_vllm_server(task="trial")

config = load_preset("default")
config.remote["enabled"] = True
config.remote["server_urls"] = ["http://localhost:8000/v1"]

trial_results, metadata, qc_report = summarize_trials(
    trials, config=config, return_metadata=True, return_qc=True
)

# Optional: stop the local vLLM server if you started it above.
# process.terminate()

### View results

In [14]:
trial_results.head()

,trial_id,clinical_space_number,clinical_space_summary,general_exclusion_criteria,space_trial_id
0,NCT03319901,1,Age range allowed: ≥60 years. Sex allowed: All...,pregnancy or breastfeeding; uncontrolled infec...,NCT03319901-1
1,NCT03319901,2,Age range allowed: ≥18 years. Sex allowed: All...,pregnancy or breastfeeding; uncontrolled infec...,NCT03319901-2
2,NCT04792489,1,Age range allowed: ≥18. Sex allowed: Male and ...,History of HIV infection; active hepatitis B s...,NCT04792489-1
3,NCT04792489,2,Age range allowed: ≥18. Sex allowed: Male and ...,History of HIV infection; active hepatitis B s...,NCT04792489-2
4,NCT04792489,3,Age range allowed: ≥18. Sex allowed: Male and ...,History of HIV infection; active hepatitis B s...,NCT04792489-3


Metadata contains a snapshot of the config used and metadata on the model used. 

In [15]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 10000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9},
   'patient': {'max_model_len': 70000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9}},
  'remote': {'enabled': True,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'chunk_size': 10000,
   'chunk_overlap'

QC report on the trial summarization run

In [16]:
qc_report

,metric,value,denominator,percent,ids
0,trials_missing_in_output,4.0,470.0,0.851064,"[NCT04115254, NCT04557449, NCT05894239, NCT062..."
1,trials_truncated_llm_response,0.0,470.0,0.000000,[]
2,spaces_exceed_embedding_token_limit,0.0,1436.0,0.000000,[]
3,spaces_per_trial_min,1.0,NaN,NaN,[]
4,spaces_per_trial_median,2.0,NaN,NaN,[]
5,spaces_per_trial_max,33.0,NaN,NaN,[]
6,trials_with_non_distinct_spaces,5.0,466.0,1.072961,"[NCT05098132, NCT05417594, NCT05458297, NCT055..."
7,spaces_dropped_missing_keyword:Age,0.0,1455.0,0.000000,[]
8,spaces_dropped_missing_keyword:Sex,0.0,1455.0,0.000000,[]
9,spaces_dropped_missing_keyword:Cancer type,0.0,1455.0,0.000000,[]


Save checked-in trial example artifacts


In [17]:
trial_results.to_csv("output/trial_summaries.csv", index=None)
with open("output/trial_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/trial_qc_report.csv", index=None)

## 2. Patient summarization walkthrough


### Read in initial notes

In [18]:
notes_path = "data/note_set_1.csv"
notes = pd.read_csv(notes_path)

### Local mode

`summarize_patients(...)` uses the local in-memory vLLM backend by default. Run this cell if you want local mode.


In [ ]:
from matchminer_ai.patients import summarize_patients

patient_summaries, metadata, qc_report = summarize_patients(
    notes, return_metadata=True, return_qc=True
)

# Optional: clear the cached local vLLM engine if GPU memory is tight.
# from matchminer_ai.llm.backends import clear_local_llm_cache
# clear_local_llm_cache()

/home/sabrina/mmai-package/src/matchminer_ai/patients/prepare.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  normalized["note_date"] = pd.to_datetime(normalized["note_date"])
Processed prompts: 100%|██████████| 1/1 [00:08<00:00,  8.99s/it, est. speed input: 553.71 toks/s, output: 137.65 toks/s]


### Remote mode

Run this cell instead of the local-mode cell if you want to send patient summarization requests to an existing OpenAI-compatible endpoint. If you do not already have a vLLM server running, `start_vllm_server(...)` can start one from the config values.


In [19]:
from matchminer_ai import load_preset
from matchminer_ai.patients import summarize_patients

os.environ["OPENAI_API_KEY"] = "not-needed"

# Optional: start a local vLLM server if you do not already have one.
# Optional: choose GPUs for the server process before starting it.
# from matchminer_ai.llm.vllm_server import start_vllm_server
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
#process = start_vllm_server(task="patient")

config = load_preset("default")
config.remote["enabled"] = True
config.remote["server_urls"] = ["http://localhost:8000/v1"]

patient_summaries, metadata, qc_report = summarize_patients(
    notes, config=config, return_metadata=True, return_qc=True
)

# Optional: stop the local vLLM server if you started it above.
#process.terminate()

/home/sabrina/mmai-package/src/matchminer_ai/patients/prepare.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  normalized["note_date"] = pd.to_datetime(normalized["note_date"])


In [22]:
patient_summaries.head()

,patient_id,last_note_date,cancer_history_summary,general_exclusion_criteria_evidence
0,11609,2022-07-21,Age: 66\nSex: Female\nCancer type: Glioblastom...,Controlled essential hypertension (recently in...
1,12019,2024-02-26,Age: 47 \nSex: Female \nCancer type: Breast ...,No untreated or progressing central nervous sy...
2,15687,2024-02-20,Age: 63 \nSex: Male \nCancer type: Small‑cel...,No evidence of common trial exclusion criteria...
3,16916000000,2024-03-06,Age: 20 \nSex: Male \nCancer type: High‑grad...,No evidence of untreated/uncontrolled brain me...
4,1722,2024-02-26,Age: 61\nSex: Male\nCancer type: Her2‑positive...,No evidence of typical exclusion criteria – no...


### Updating existing patient summaries

Run this optional cell after the local- or remote-mode cell above if you want to simulate a longitudinal update. It reads a second set of new notes, uses the existing `patient_summaries` as the starting point, then updates those summaries with the new notes.


In [ ]:
from matchminer_ai.patients import summarize_patients

new_notes_path = "data/note_set_2.csv"
new_notes = pd.read_csv(new_notes_path)

existing_summaries = patient_summaries.assign(
    patient_summary=lambda df: (
        df["cancer_history_summary"].fillna("")
        + "\n\nBoilerplate:\n"
        + df["general_exclusion_criteria_evidence"].fillna("")
    )
)[["patient_id", "patient_summary"]]

patient_summaries, metadata, qc_report = summarize_patients(
    new_notes,
    config=config,
    existing_summaries=existing_summaries,
    return_metadata=True,
    return_qc=True,
)

### View results

In [25]:
patient_summaries

,patient_id,last_note_date,cancer_history_summary,general_exclusion_criteria_evidence
0,11609,2026-05-03,Age: 72\nSex: Female\nCancer type: Glioblastom...,Chronic uncontrolled hypertension historically...
1,12019,2026-05-14,Age: 52 \nSex: Female \nCancer type: Breast ...,ECOG 1; no untreated or evolving CNS metastase...
2,15687,2025-10-07,Age: 68\nSex: Male\nCancer type: Small‑cell lu...,• No untreated/uncontrolled brain metastases; ...
3,16916000000,2024-11-14,Age: 21 \nSex: Male \nCancer type: High‑grad...,"No untreated or uncontrolled brain metastases,..."
4,1722,2025-02-10,Age: 64\nSex: Male\nCancer type: HER2‑positive...,No uncontrolled central nervous system disease...
5,20838,2026-04-30,Age: 70\nSex: Male\nCancer type: Prostate aden...,Controlled hypertension on amlodipide 5 mg dai...
6,21339,2026-05-10,Age: 76\nSex: Female\nCancer type: High‑grade ...,No evidence of common trial exclusion criteria...
7,3996000000,2026-01-04,Age: 46\nSex: Male\nCancer type: T-cell lympho...,- Ongoing grade 2 peripheral sensory neuropath...
8,4030000000,2025-10-18,Age: 55\nSex: Female\nCancer type: Ovarian can...,No evidence of common exclusion criteria: no C...
9,4716,2026-01-17,Age: 34\nSex: Male\nCancer type: Appendiceal w...,No uncontrolled comorbidities – hypertension c...


Metadata contains a snapshot of the config used and metadata on the models used. 

In [26]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 10000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9},
   'patient': {'max_model_len': 70000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9}},
  'remote': {'enabled': True,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'chunk_size': 10000,
   'chunk_overlap'

QC report on patient summarization

In [27]:
qc_report

,metric,value,denominator,percent,ids
0,patients_dropped_noninformative_summary,0,12,0.0,[]
1,patients_exceed_embedding_token_limit,0,12,0.0,[]
2,patients_exclusion_criteria_not_extracted,0,12,0.0,[]
3,patients_missing_keyword:Age,0,12,0.0,[]
4,patients_missing_keyword:Sex,0,12,0.0,[]
5,patients_missing_keyword:Cancer type,0,12,0.0,[]
6,patients_missing_keyword:Histology,0,12,0.0,[]
7,patients_missing_keyword:Current extent,0,12,0.0,[]
8,patients_missing_keyword:Biomarkers,0,12,0.0,[]
9,patients_missing_keyword:Treatment history,0,12,0.0,[]


Save checked-in patient example artifacts

In [28]:
patient_summaries.to_csv("output/patient_summaries.csv", index=None)
with open("output/patient_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/patient_qc_report.csv", index=None)

## 3. Embed summarized entities for matching


In [29]:
from matchminer_ai.embedding import embed_for_matching

In [15]:
# Uncomment if you want to start from previously generated summaries
# trial_results = pd.read_csv("output/trial_summaries.csv")
# patient_summaries = pd.read_csv("output/patient_summaries.csv", dtype="str")

### Trials

In [30]:
embedded_trials, metadata = embed_for_matching(
    trial_results, entity_type="trial", return_metadata=True
)

In [31]:
embedded_trials.head()

,space_trial_id,embedding
0,NCT03319901-1,"[0.032892655581235886, 0.030737916007637978, -..."
1,NCT03319901-2,"[0.01636856235563755, 0.013839730992913246, -0..."
2,NCT04792489-1,"[-0.04821357503533363, -0.02354833111166954, -..."
3,NCT04792489-2,"[-0.037800826132297516, -0.013209248892962933,..."
4,NCT04792489-3,"[-0.04784146696329117, -0.018136244267225266, ..."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [32]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 10000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9},
   'patient': {'max_model_len': 70000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'chunk_size': 10000,
   'chunk_overlap

### Patients

In [33]:
embedded_patients, metadata = embed_for_matching(
    patient_summaries, entity_type="patient", return_metadata=True
)

In [34]:
embedded_patients.head()

,patient_id,embedding
0,11609,"[-0.027026936411857605, -0.03773500770330429, ..."
1,12019,"[-0.03157830238342285, -0.010270429775118828, ..."
2,15687,"[-0.005254311952739954, -0.007536736316978931,..."
3,16916000000,"[0.027958551421761513, -0.021866077557206154, ..."
4,1722,"[0.05782292038202286, -0.03596945479512215, -0..."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [35]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 10000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9},
   'patient': {'max_model_len': 70000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'chunk_size': 10000,
   'chunk_overlap

In [36]:
# Optional local export if you want to reuse embeddings without rerunning this step.
embedded_patients.to_parquet("output/embedded_patients.parquet")
embedded_trials.to_parquet("output/embedded_trials.parquet")

## 4. Generate candidate matches

Generate ranked trial-space candidates for each patient.


In [37]:
from matchminer_ai.matching import generate_candidate_matches

In [38]:
# Uncomment if you saved embeddings locally and want to start from them
# embedded_patients = pd.read_parquet('output/embedded_patients.parquet')
# embedded_trials = pd.read_parquet('output/embedded_trials.parquet')

### Candidate matches


In [39]:
candidate_matches = generate_candidate_matches(
    query_df=embedded_patients, corpus_df=embedded_trials
)

In [40]:
candidate_matches.head()

,patient_id,space_trial_id,similarity_score,rank
0,11609,NCT06586957-7,0.457802,1
1,11609,NCT05538130-3,0.448013,2
2,11609,NCT03423628-1,0.438403,3
3,11609,NCT06001749-1,0.438086,4
4,11609,NCT06641908-2,0.437979,5


## 5. Match quality check


In [41]:
from matchminer_ai.matching import score_match_quality

Add back the patient and trial summary context required by the match-quality checker.


In [42]:
candidate_matches_with_context = (
    candidate_matches.merge(
        patient_summaries[["patient_id", "cancer_history_summary"]],
        on="patient_id",
        how="left",
    ).merge(
        trial_results[["space_trial_id", "clinical_space_summary"]],
        on="space_trial_id",
        how="left",
    )
)
candidate_matches_with_context.head()

,patient_id,space_trial_id,similarity_score,rank,cancer_history_summary,clinical_space_summary
0,11609,NCT06586957-7,0.457802,1,Age: 72\nSex: Female\nCancer type: Glioblastom...,Age range allowed: ≥18 years. Sex allowed: Mal...
1,11609,NCT05538130-3,0.448013,2,Age: 72\nSex: Female\nCancer type: Glioblastom...,Age range allowed: NA. Sex allowed: Both. Canc...
2,11609,NCT03423628-1,0.438403,3,Age: 72\nSex: Female\nCancer type: Glioblastom...,Age range allowed: Any. Sex allowed: Both sexe...
3,11609,NCT06001749-1,0.438086,4,Age: 72\nSex: Female\nCancer type: Glioblastom...,Age range allowed: 18 years and older. Sex all...
4,11609,NCT06641908-2,0.437979,5,Age: 72\nSex: Female\nCancer type: Glioblastom...,Age range allowed: NA. Sex allowed: All. Cance...


Run the match-quality checker.


In [43]:
match_quality_matches, metadata = score_match_quality(
    candidate_matches_with_context, return_metadata=True
)

Loading weights: 100%|██████████| 174/174 [00:00<00:00, 790.87it/s]


In [44]:
match_quality_matches.head()

,patient_id,space_trial_id,match_quality_score,match_quality_pass
0,11609,NCT06586957-7,0.378301,True
1,11609,NCT05538130-3,0.546338,True
2,11609,NCT03423628-1,0.770821,True
3,11609,NCT06001749-1,0.593715,True
4,11609,NCT06641908-2,0.961645,True


In [45]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 10000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9},
   'patient': {'max_model_len': 70000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'chunk_size': 10000,
   'chunk_overlap

## 6. Exclusion criteria check


In [46]:
# limit to unique patient-trial ID pairs
match_quality_matches["trial_id"] = (
    match_quality_matches["space_trial_id"].str.split("-").str[0]
)
unique_patient_trial_pairs = match_quality_matches[
    ["patient_id", "trial_id"]
].drop_duplicates()

# add in exclusion criteria for patients and trials
patient_trial_pairs_with_exclusion_context = unique_patient_trial_pairs.merge(
    patient_summaries[["patient_id", "general_exclusion_criteria_evidence"]],
    on="patient_id",
    how="left",
).merge(
    trial_results[["trial_id", "general_exclusion_criteria"]].drop_duplicates(),
    on="trial_id",
    how="left",
)
patient_trial_pairs_with_exclusion_context.head()

,patient_id,trial_id,general_exclusion_criteria_evidence,general_exclusion_criteria
0,11609,NCT06586957,Chronic uncontrolled hypertension historically...,History of another malignant neoplasm except a...
1,11609,NCT05538130,Chronic uncontrolled hypertension historically...,Brain metastasis larger than 4 cm.\nHistory or...
2,11609,NCT03423628,Chronic uncontrolled hypertension historically...,History of severe traumatic brain injury or ce...
3,11609,NCT06001749,Chronic uncontrolled hypertension historically...,Participants unable to give adequate informed ...
4,11609,NCT06641908,Chronic uncontrolled hypertension historically...,History of malignant solid tumors (other than ...


In [47]:
from matchminer_ai.matching import exclusion_criteria_check

# run exclusion criteria check
exclusion_results, metadata = exclusion_criteria_check(
    patient_trial_pairs_with_exclusion_context, return_metadata=True
)
exclusion_results.head()

Loading weights: 100%|██████████| 174/174 [00:00<00:00, 823.78it/s]


,patient_id,trial_id,exclusion_score,exclusion_criteria_pass
0,11609,NCT06586957,0.969870,True
1,11609,NCT05538130,0.999236,True
2,11609,NCT03423628,0.972369,True
3,11609,NCT06001749,0.975016,False
4,11609,NCT06641908,0.998581,True


In [48]:
exclusion_results["exclusion_criteria_pass"].value_counts()

exclusion_criteria_pass
True     153
False     10
Name: count, dtype: int64

In [49]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 10000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9},
   'patient': {'max_model_len': 70000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'max_concurrent_requests': 16,
   'request_timeout': 600,
   'max_retries': 6,
   'batch_size': 1000,
   'retry_backoff_base': 1.0},
  'trial': {'model_name': 'openai/gpt-oss-120b',
   'sampling_params': {'temperature': 0.0,
    'top_k': 1,
    'max_tokens': 7500,
    'repetition_penalty': 1.3},
   'prompt_files': {'primer': 'trial.user.primer.txt',
    'question': 'trial.user.question.txt'},
   'reasoning_marker': 'assistantfinal',
   'boilerplate_marker': '\\n.*Boilerplate.*\\n'},
  'patient': {'model_name': 'openai/gpt-oss-120b',
   'chunk_size': 10000,
   'chunk_overlap

## 7. Save final matches

Create one review table with one row per patient/trial pair. Each pair is represented by its top-scoring clinical space, rows are ranked by that top match-quality score, and exclusion-check results are included as labels without filtering rows.


In [51]:
patient_context_cols = [
    col
    for col in [
        "patient_id",
        "cancer_history_summary",
        "general_exclusion_criteria_evidence",
    ]
    if col in patient_summaries.columns
]
trial_context_cols = [
    col
    for col in [
        "space_trial_id",
        "clinical_space_number",
        "clinical_space_summary",
        "general_exclusion_criteria",
    ]
    if col in trial_results.columns
]
original_trial_cols = [
    col
    for col in [
        "trial_id",
        "oncore_id",
        "trial_title",
        "brief_summary",
        "detailed_summary",
        "eligibility_criteria",
    ]
    if col in trials.columns
]

final_matches = (
    match_quality_matches.merge(
        candidate_matches[["patient_id", "space_trial_id", "similarity_score", "rank"]],
        on=["patient_id", "space_trial_id"],
        how="left",
    )
    .merge(
        exclusion_results,
        on=["patient_id", "trial_id"],
        how="left",
    )
    .merge(
        patient_summaries[patient_context_cols],
        on="patient_id",
        how="left",
    )
    .merge(
        trial_results[trial_context_cols].drop_duplicates("space_trial_id"),
        on="space_trial_id",
        how="left",
    )
    .merge(
        trials[original_trial_cols].drop_duplicates("trial_id"),
        on="trial_id",
        how="left",
    )
)

# Keep one row per patient/trial pair, represented by the top-scoring clinical space.
final_matches = (
    final_matches.sort_values(
        ["patient_id", "trial_id", "match_quality_score"],
        ascending=[True, True, False],
    )
    .drop_duplicates(["patient_id", "trial_id"], keep="first")
    .sort_values(["patient_id", "match_quality_score"], ascending=[True, False])
    .reset_index(drop=True)
)
final_matches["match_quality_rank"] = final_matches.groupby("patient_id").cumcount() + 1

final_matches = final_matches.rename(
    columns={
        "match_quality_rank": "match_rank",
        "exclusion_criteria_pass": "passes_exclusion_criteria",
        "cancer_history_summary": "patient_summary",
        "general_exclusion_criteria_evidence": "patient_exclusion_evidence",
        "brief_summary": "trial_brief_summary",
        "detailed_summary": "trial_detailed_summary",
        "eligibility_criteria": "trial_eligibility_criteria",
        "clinical_space_summary": "matched_clinical_space_summary",
        "general_exclusion_criteria": "extracted_trial_exclusion_criteria",
    }
)

ordered_cols = [
    "patient_id",
    "trial_id",
    "match_rank",
    "passes_exclusion_criteria",
    "patient_summary",
    "patient_exclusion_evidence",
    "trial_title",
    "trial_brief_summary",
    "trial_eligibility_criteria",
    "clinical_space_number",
    "matched_clinical_space_summary",
    "extracted_trial_exclusion_criteria",
]
final_matches = final_matches[[col for col in ordered_cols if col in final_matches.columns]]
final_matches.head()


,patient_id,trial_id,match_rank,passes_exclusion_criteria,patient_summary,patient_exclusion_evidence,trial_title,trial_brief_summary,trial_eligibility_criteria,clinical_space_number,matched_clinical_space_summary,extracted_trial_exclusion_criteria
0,11609,NCT06641908,1,True,Age: 72\nSex: Female\nCancer type: Glioblastom...,Chronic uncontrolled hypertension historically...,"A Phase 1, Two-Part, Multicenter, Open-Label F...",The purpose of this study is to establish the ...,Inclusion Criteria:\n\n* Escalation A: partici...,2,Age range allowed: NA. Sex allowed: All. Cance...,History of malignant solid tumors (other than ...
1,11609,NCT06810544,2,False,Age: 72\nSex: Female\nCancer type: Glioblastom...,Chronic uncontrolled hypertension historically...,"A Phase 1/2, Multicenter, Open-Label Study to ...",This is a first in human study of TNG456 alone...,Inclusion Criteria:\n\n* Has a tumor with a co...,1,Age range allowed: ≥18 years. Sex allowed: bot...,History of symptomatic interstitial pneumonia/...
2,11609,NCT03423628,3,True,Age: 72\nSex: Female\nCancer type: Glioblastom...,Chronic uncontrolled hypertension historically...,"A Phase I, Multicentre Study to Assess the Saf...",This study will test an investigational drug c...,Inclusion Criteria:\n\n* Provision of formalin...,1,Age range allowed: Any. Sex allowed: Both sexe...,History of severe traumatic brain injury or ce...
3,11609,NCT04794699,4,True,Age: 72\nSex: Female\nCancer type: Glioblastom...,Chronic uncontrolled hypertension historically...,"An Open Label, Phase 1, Treatment Study to Eva...","This is a Phase 1, open-label, multicenter, do...",Inclusion Criteria:\n\n* Participant must be a...,1,Age range allowed: >=18. Sex allowed: Male and...,Known symptomatic brain metastases\nKnown prim...
4,11609,NCT06589596,5,True,Age: 72\nSex: Female\nCancer type: Glioblastom...,Chronic uncontrolled hypertension historically...,"A Phase 1a/b Study Investigating the Safety, T...","This is an open-label, multicenter, first-in-h...",Inclusion Criteria:\n\n* Participants must sig...,1,Age range allowed: NA. Sex allowed: All. Cance...,Active leptomeningeal disease or symptomatic s...


In [52]:
# Optional local export for review or downstream analysis.
final_matches.to_csv("output/final_matches.csv", index=None)